In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/alexandr2017/training-set/training_set.jsonl
/kaggle/input/datasets/alexandr2017/eval-set/eval_set.jsonl


# LESSON 17: LLM Fine-Tuning in Production
## ShopFlow Support Email Classifier

In [2]:
# ────────────────────────────────────────────────────────────
# 1: Встановлення бібліотек
# ~5 хвилин
# ────────────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "-q",
    "unsloth[kaggle-new]", "unsloth_zoo",
    "bitsandbytes", "trl", "peft",
    "torchao>=0.16.0", "cut_cross_entropy",
    "hf_transfer", "msgspec",
    "--upgrade"
], check=False)
 
print("✅ Бібліотеки встановлено")


✅ Бібліотеки встановлено


In [3]:
# ────────────────────────────────────────────────────────────
# 2: Середовище + імпорти
# ────────────────────────────────────────────────────────────
import os, json, re, hashlib, torch
from collections import defaultdict, Counter
from kaggle_secrets import UserSecretsClient
 
# HF токен
secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
 
# Робоча папка
BASE_DIR = '/kaggle/working/lesson17_finetune'
os.makedirs(BASE_DIR, exist_ok=True)
 
# Шляхи до датасетів
EVAL_PATH  = '/kaggle/input/datasets/alexandr2017/eval-set/eval_set.jsonl'
TRAIN_PATH = '/kaggle/input/datasets/alexandr2017/training-set/training_set.jsonl'
 
# Фіксований промпт — однаковий для baseline і fine-tuned
SYSTEM_PROMPT = """You are a support email classifier. Extract information from the email and return ONLY a valid JSON object with exactly these fields:
- sender: full name of the sender (null if anonymous)
- product: one of [Payments, Inventory, Analytics, Storefront, Shipping, Subscriptions]
- category: one of [billing, technical, account, feature_request, other]
- urgency: one of [low, medium, high, critical]
- summary: one sentence summary of the issue
 
Return ONLY the JSON object, no explanation, no markdown, no extra text."""
 
def build_prompt(email_text):
    return f"{SYSTEM_PROMPT}\n\nEmail:\n{email_text}\n\nJSON:"
 
def parse_json_response(raw):
    raw = raw.strip()
    try:
        return json.loads(raw)
    except:
        pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except: pass
    match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
    if match:
        try: return json.loads(match.group(0))
        except: pass
    return None
 
def score_prediction(pred, expected):
    if pred is None:
        return {k: 0 for k in ['json_valid','sender_correct','product_correct',
                                'category_correct','urgency_correct',
                                'summary_present','exact_match']}
    sender_correct = (
        (pred.get('sender') is None and expected.get('sender') is None) or
        (str(pred.get('sender','')).strip().lower() ==
         str(expected.get('sender','')).strip().lower())
    )
    product_correct  = pred.get('product')  == expected.get('product')
    category_correct = pred.get('category') == expected.get('category')
    urgency_correct  = pred.get('urgency')  == expected.get('urgency')
    summary_present  = bool(pred.get('summary','').strip())
    exact_match = all([sender_correct, product_correct,
                       category_correct, urgency_correct, summary_present])
    return {
        'json_valid'       : 1,
        'sender_correct'   : int(sender_correct),
        'product_correct'  : int(product_correct),
        'category_correct' : int(category_correct),
        'urgency_correct'  : int(urgency_correct),
        'summary_present'  : int(summary_present),
        'exact_match'      : int(exact_match),
    }
 
import unsloth, bitsandbytes as bnb, transformers
print(f"transformers : {transformers.__version__}")
print(f"bitsandbytes : {bnb.__version__}")
print(f"unsloth      : {unsloth.__version__}")
print(f"✅ Середовище готове")
 


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
transformers : 5.5.0
bitsandbytes : 0.49.2
unsloth      : 2026.6.1
✅ Середовище готове


In [4]:
# ────────────────────────────────────────────────────────────
# 3: Завантаження та валідація eval set
# ────────────────────────────────────────────────────────────
REQUIRED_FIELDS  = {"sender", "product", "category", "urgency", "summary"}
VALID_CATEGORIES = {"billing", "technical", "account", "feature_request", "other"}
VALID_URGENCIES  = {"low", "medium", "high", "critical"}
VALID_PRODUCTS   = {"Payments", "Inventory", "Analytics",
                    "Storefront", "Shipping", "Subscriptions"}
 
errors, hashes = [], {}
with open(EVAL_PATH, 'r', encoding='utf-8') as f:
    eval_data = [json.loads(line) for line in f]
 
for item in eval_data:
    eid = item.get("id", "?")
    exp = item.get("expected", {})
    if exp.get("category") not in VALID_CATEGORIES:
        errors.append(f"{eid}: invalid category")
    if exp.get("urgency") not in VALID_URGENCIES:
        errors.append(f"{eid}: invalid urgency")
    h = hashlib.md5(item['email'].strip().encode()).hexdigest()
    if h in hashes:
        errors.append(f"{eid}: duplicate")
    hashes[h] = eid
 
# Зберігаємо хеші для hash-check
with open(os.path.join(BASE_DIR, 'eval_hashes.json'), 'w') as f:
    json.dump(hashes, f)
 
urgencies  = Counter(item['expected']['urgency']  for item in eval_data)
categories = Counter(item['expected']['category'] for item in eval_data)
anon_count = sum(1 for item in eval_data if item['expected']['sender'] is None)
 
print(f"{'='*45}")
print(f"EVAL SET ВАЛІДАЦІЯ")
print(f"{'='*45}")
print(f"Прикладів : {len(eval_data)} | Помилок: {len(errors)}")
print(f"Anonymous : {anon_count}")
print(f"\nURGENCY: {dict(urgencies)}")
print(f"CATEGORY: {dict(categories)}")
if errors:
    print(f"\n🔴 ПОМИЛКИ: {errors}")
else:
    print(f"\n✅ Eval set валідний!")
 


EVAL SET ВАЛІДАЦІЯ
Прикладів : 30 | Помилок: 0
Anonymous : 3

URGENCY: {'critical': 7, 'high': 9, 'low': 10, 'medium': 4}
CATEGORY: {'technical': 14, 'billing': 6, 'feature_request': 5, 'account': 5}

✅ Eval set валідний!


In [5]:
# ────────────────────────────────────────────────────────────
# 4: Baseline — завантаження Llama 3.1 8B
# ~5-7 хвилин, ~6GB GPU RAM
# ────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
 
MODEL_NAME = 'meta-llama/Llama-3.1-8B'
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
 
print(f"⏳ Завантажуємо {MODEL_NAME} для baseline...")
 
base_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, token=HF_TOKEN)
 
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)
base_model.eval()
 
allocated = torch.cuda.memory_allocated() / 1024**3
print(f"✅ Baseline модель завантажена! GPU RAM: {allocated:.1f} GB")
 


⏳ Завантажуємо meta-llama/Llama-3.1-8B для baseline...


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

✅ Baseline модель завантажена! GPU RAM: 1.4 GB


In [6]:
# ────────────────────────────────────────────────────────────
# 5: Baseline evaluation
# ~10-15 хвилин
# ────────────────────────────────────────────────────────────
def run_baseline_inference(email_text, max_new_tokens=256):
    prompt  = build_prompt(email_text)
    inputs  = base_tokenizer(prompt, return_tensors='pt').to(base_model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = base_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][input_len:]
    return base_tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)
 
baseline_results = []
total_tokens     = 0
 
print(f"🚀 Baseline evaluation на {len(eval_data)} прикладах...\n")
 
for i, item in enumerate(eval_data, 1):
    raw, tokens = run_baseline_inference(item['email'])
    pred        = parse_json_response(raw)
    scores      = score_prediction(pred, item['expected'])
    total_tokens += tokens
    baseline_results.append({
        'id': item['id'], 'raw': raw, 'parsed': pred,
        'expected': item['expected'], 'scores': scores, 'tokens': tokens,
    })
    status     = '✅' if scores['json_valid']      else '❌'
    urgency_ok = '✅' if scores['urgency_correct'] else '❌'
    print(f"  {i:2}/30 {item['id']} | JSON:{status} | urgency:{urgency_ok} "
          f"| tokens:{tokens:3} | pred_urgency: {pred.get('urgency','—') if pred else 'NONE'}")
 
# Зберігаємо результати
with open(os.path.join(BASE_DIR, 'baseline_results.json'), 'w') as f:
    json.dump(baseline_results, f, ensure_ascii=False, indent=2)
print(f"\n💾 Baseline результати збережено")
 


🚀 Baseline evaluation на 30 прикладах...

   1/30 eval_001 | JSON:✅ | urgency:❌ | tokens:256 | pred_urgency: high
   2/30 eval_002 | JSON:✅ | urgency:✅ | tokens: 56 | pred_urgency: high
   3/30 eval_003 | JSON:✅ | urgency:✅ | tokens: 46 | pred_urgency: low
   4/30 eval_004 | JSON:✅ | urgency:❌ | tokens: 44 | pred_urgency: high
   5/30 eval_005 | JSON:✅ | urgency:✅ | tokens:256 | pred_urgency: low
   6/30 eval_006 | JSON:✅ | urgency:✅ | tokens: 50 | pred_urgency: high
   7/30 eval_007 | JSON:✅ | urgency:✅ | tokens:256 | pred_urgency: low
   8/30 eval_008 | JSON:✅ | urgency:❌ | tokens: 45 | pred_urgency: low
   9/30 eval_009 | JSON:✅ | urgency:❌ | tokens:256 | pred_urgency: high
  10/30 eval_010 | JSON:✅ | urgency:✅ | tokens:256 | pred_urgency: low
  11/30 eval_011 | JSON:✅ | urgency:✅ | tokens: 48 | pred_urgency: critical
  12/30 eval_012 | JSON:✅ | urgency:❌ | tokens: 47 | pred_urgency: low
  13/30 eval_013 | JSON:✅ | urgency:❌ | tokens: 43 | pred_urgency: low
  14/30 eval_014 | JSON:✅

In [7]:
# ────────────────────────────────────────────────────────────
# 6: Baseline метрики
# ────────────────────────────────────────────────────────────
n      = len(baseline_results)
totals = defaultdict(int)
for r in baseline_results:
    for k, v in r['scores'].items():
        totals[k] += v
 
avg_tokens_base = total_tokens / n
 
print(f"\n{'='*55}")
print(f"  BASELINE — Llama 3.1 8B (без fine-tuning)")
print(f"{'='*55}")
print(f"  {'Метрика':<25} {'Правильно':>10} {'%':>8}")
print(f"  {'-'*45}")
for label, key in [
    ('json_valid',       'json_valid'),
    ('exact_match',      'exact_match'),
    ('sender accuracy',  'sender_correct'),
    ('product accuracy', 'product_correct'),
    ('category accuracy','category_correct'),
    ('urgency accuracy', 'urgency_correct'),
    ('summary present',  'summary_present'),
]:
    print(f"  {label:<25} {totals[key]:>10}/{n} {totals[key]/n*100:>7.1f}%")
print(f"  {'-'*45}")
print(f"  {'avg tokens':<25} {avg_tokens_base:>10.1f}")
print(f"{'='*55}")
 
baseline_metrics = {
    "model"             : "Llama-3.1-8B-base",
    "json_valid"        : round(totals['json_valid']/n*100, 1),
    "exact_match"       : round(totals['exact_match']/n*100, 1),
    "sender_accuracy"   : round(totals['sender_correct']/n*100, 1),
    "product_accuracy"  : round(totals['product_correct']/n*100, 1),
    "category_accuracy" : round(totals['category_correct']/n*100, 1),
    "urgency_accuracy"  : round(totals['urgency_correct']/n*100, 1),
    "summary_present"   : round(totals['summary_present']/n*100, 1),
    "avg_tokens"        : round(avg_tokens_base, 1),
}
with open(os.path.join(BASE_DIR, 'baseline_metrics.json'), 'w') as f:
    json.dump(baseline_metrics, f, indent=2)
print(f"\n✅ baseline_metrics.json збережено")
 
# Звільняємо GPU пам'ять перед fine-tuning
del base_model
torch.cuda.empty_cache()
print(f"✅ GPU пам'ять звільнено для fine-tuning")
 



  BASELINE — Llama 3.1 8B (без fine-tuning)
  Метрика                    Правильно        %
  ---------------------------------------------
  json_valid                        30/30   100.0%
  exact_match                       10/30    33.3%
  sender accuracy                   27/30    90.0%
  product accuracy                  19/30    63.3%
  category accuracy                 23/30    76.7%
  urgency accuracy                  19/30    63.3%
  summary present                   30/30   100.0%
  ---------------------------------------------
  avg tokens                     125.1

✅ baseline_metrics.json збережено
✅ GPU пам'ять звільнено для fine-tuning


In [8]:
# ────────────────────────────────────────────────────────────
# 7: Fine-tuning — завантаження через Unsloth
# ~3-5 хвилин
# ────────────────────────────────────────────────────────────
from unsloth import FastLanguageModel, is_bfloat16_supported
 
MAX_SEQ_LENGTH = 1024
LORA_R         = 16
LORA_ALPHA     = 32
 
print(f"⏳ Завантажуємо {MODEL_NAME} через Unsloth...")
 
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)
 
ft_model = FastLanguageModel.get_peft_model(
    ft_model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0.0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
 
total_p    = sum(p.numel() for p in ft_model.parameters())
trainable  = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
print(f"✅ Модель завантажена!")
print(f"   Навчальних параметрів: {trainable:,} ({trainable/total_p*100:.2f}%)")
print(f"   GPU RAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
 


⏳ Завантажуємо meta-llama/Llama-3.1-8B через Unsloth...
==((====))==  Unsloth 2026.6.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Модель завантажена!
   Навчальних параметрів: 41,943,040 (0.90%)
   GPU RAM: 5.8 GB


In [9]:
# ────────────────────────────────────────────────────────────
# 8: Підготовка training set
# ────────────────────────────────────────────────────────────
from datasets import Dataset
 
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    train_data = [json.loads(line) for line in f]
 
# Hash-check: перевіряємо що training set не перетинається з eval set
eval_hashes = set(json.load(open(os.path.join(BASE_DIR, 'eval_hashes.json'))).keys())
overlaps = 0
for item in train_data:
    h = hashlib.md5(item['email'].strip().encode()).hexdigest()
    if h in eval_hashes:
        overlaps += 1
        print(f"⚠️ OVERLAP: {item['id']}")
 
print(f"✅ Hash-check: {len(train_data)} прикладів, {overlaps} overlap з eval set")
 
def format_example(item):
    prompt    = build_prompt(item['email'])
    output    = json.dumps(item['expected'], ensure_ascii=False)
    full_text = prompt + " " + output + ft_tokenizer.eos_token
    return {"text": full_text}
 
formatted = [format_example(item) for item in train_data]
dataset   = Dataset.from_list(formatted)
print(f"✅ Датасет готовий: {len(dataset)} прикладів")
 


✅ Hash-check: 300 прикладів, 0 overlap з eval set
✅ Датасет готовий: 300 прикладів


In [10]:
# ────────────────────────────────────────────────────────────
# 9: Fine-tuning
# ~15-30 хвилин на T4
# ────────────────────────────────────────────────────────────
from unsloth import UnslothTrainer, UnslothTrainingArguments
 
EPOCHS        = 3
BATCH_SIZE    = 2
GRAD_ACCUM    = 8
LEARNING_RATE = 2e-4
 
OUTPUT_DIR = os.path.join(BASE_DIR, 'checkpoints')
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
trainer = UnslothTrainer(
    model              = ft_model,
    tokenizer          = ft_tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    args = UnslothTrainingArguments(
        output_dir                  = OUTPUT_DIR,
        num_train_epochs            = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate               = LEARNING_RATE,
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 5,
        save_strategy               = "no",
        warmup_steps                = 10,
        lr_scheduler_type           = "cosine",
        optim                       = "adamw_8bit",
        seed                        = 42,
        report_to                   = "none",
    ),
)
 
print(f"🚀 Fine-tuning...")
print(f"   Epochs: {EPOCHS} | LR: {LEARNING_RATE} | Batch: {BATCH_SIZE*GRAD_ACCUM}\n")
 
trainer_stats = trainer.train()
 
print(f"\n✅ Fine-tuning завершено!")
print(f"   Час : {trainer_stats.metrics['train_runtime']/60:.1f} хв")
print(f"   Loss: {trainer_stats.metrics['train_loss']:.4f}")
 


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/300 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
🚀 Fine-tuning...
   Epochs: 3 | LR: 0.0002 | Batch: 16



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 300 | Num Epochs = 3 | Total steps = 57
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.137000
10,1.328914
15,0.629100
20,0.539082
25,0.491047
30,0.459580
35,0.453952
40,0.419708
45,0.403679
50,0.361278



✅ Fine-tuning завершено!
   Час : 12.7 хв
   Loss: 0.6792


In [11]:
# ────────────────────────────────────────────────────────────
# 10: Збереження адаптера
# ────────────────────────────────────────────────────────────
ADAPTER_DIR = os.path.join(BASE_DIR, 'lora_adapter')
os.makedirs(ADAPTER_DIR, exist_ok=True)
 
ft_model.save_pretrained(ADAPTER_DIR)
ft_tokenizer.save_pretrained(ADAPTER_DIR)
 
total_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(ADAPTER_DIR)
    for f in files
)
print(f"✅ Адаптер збережено → {ADAPTER_DIR}")
print(f"   Розмір: {total_size/1024/1024:.1f} MB")
 


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lesson17_finetune/lora_adapter/tokenizer_config.json.


✅ Адаптер збережено → /kaggle/working/lesson17_finetune/lora_adapter
   Розмір: 176.5 MB


In [12]:
# ────────────────────────────────────────────────────────────
# 11: Re-evaluation fine-tuned моделі
# ~5-10 хвилин
# ────────────────────────────────────────────────────────────
FastLanguageModel.for_inference(ft_model)
print("✅ Модель в режимі інференсу\n")
 
def run_ft_inference(email_text, max_new_tokens=256):
    prompt    = build_prompt(email_text)
    inputs    = ft_tokenizer(prompt, return_tensors='pt').to(ft_model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = ft_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][input_len:]
    return ft_tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)
 
ft_results   = []
total_tokens = 0
 
print(f"🚀 Re-evaluation на {len(eval_data)} прикладах...\n")
 
for i, item in enumerate(eval_data, 1):
    raw, tokens = run_ft_inference(item['email'])
    pred        = parse_json_response(raw)
    scores      = score_prediction(pred, item['expected'])
    total_tokens += tokens
    ft_results.append({
        'id': item['id'], 'raw': raw, 'parsed': pred,
        'expected': item['expected'], 'scores': scores, 'tokens': tokens,
    })
    status     = '✅' if scores['json_valid']      else '❌'
    urgency_ok = '✅' if scores['urgency_correct'] else '❌'
    pred_urg   = pred.get('urgency','—') if pred else 'NONE'
    exp_urg    = item['expected']['urgency']
    print(f"  {i:2}/30 {item['id']} | JSON:{status} | urgency:{urgency_ok} "
          f"| pred:{pred_urg:8} | expected:{exp_urg}")
 
with open(os.path.join(BASE_DIR, 'ft_results.json'), 'w') as f:
    json.dump(ft_results, f, ensure_ascii=False, indent=2)
print(f"\n💾 FT результати збережено")
 


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


✅ Модель в режимі інференсу

🚀 Re-evaluation на 30 прикладах...



Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   1/30 eval_001 | JSON:✅ | urgency:✅ | pred:critical | expected:critical


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

  27/30 eval_027 | JSON:✅ | urgency:✅ | pred:high     | expected:high


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  28/30 eval_028 | JSON:✅ | urgency:❌ | pred:medium   | expected:low


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  29/30 eval_029 | JSON:✅ | urgency:✅ | pred:critical | expected:critical
  30/30 eval_030 | JSON:✅ | urgency:✅ | pred:low      | expected:low

💾 FT результати збережено


In [13]:
# ────────────────────────────────────────────────────────────
# КЛІТИНКА 12: Фінальна таблиця порівняння
# ────────────────────────────────────────────────────────────
n      = len(ft_results)
totals = defaultdict(int)
for r in ft_results:
    for k, v in r['scores'].items():
        totals[k] += v
 
avg_tokens_ft = total_tokens / n
 
ft_metrics = {
    "model"             : "Llama-3.1-8B-finetuned",
    "json_valid"        : round(totals['json_valid']/n*100, 1),
    "exact_match"       : round(totals['exact_match']/n*100, 1),
    "sender_accuracy"   : round(totals['sender_correct']/n*100, 1),
    "product_accuracy"  : round(totals['product_correct']/n*100, 1),
    "category_accuracy" : round(totals['category_correct']/n*100, 1),
    "urgency_accuracy"  : round(totals['urgency_correct']/n*100, 1),
    "summary_present"   : round(totals['summary_present']/n*100, 1),
    "avg_tokens"        : round(avg_tokens_ft, 1),
}
with open(os.path.join(BASE_DIR, 'ft_metrics.json'), 'w') as f:
    json.dump(ft_metrics, f, indent=2)
 
# Завантажуємо baseline для порівняння
with open(os.path.join(BASE_DIR, 'baseline_metrics.json')) as f:
    bm = json.load(f)
 
metrics_order = [
    ('json_valid',        'json_valid'),
    ('exact_match',       'exact_match'),
    ('sender accuracy',   'sender_accuracy'),
    ('product accuracy',  'product_accuracy'),
    ('category accuracy', 'category_accuracy'),
    ('urgency accuracy',  'urgency_accuracy'),
    ('summary present',   'summary_present'),
]
 
print(f"\n{'='*65}")
print(f"  ФІНАЛЬНЕ ПОРІВНЯННЯ: Base 8B vs Fine-tuned 8B")
print(f"{'='*65}")
print(f"  {'Метрика':<22} {'Base':>8} {'FT':>8} {'Δ':>8}")
print(f"  {'-'*55}")
 
for label, key in metrics_order:
    b   = bm[key]
    ft  = ft_metrics[key]
    d   = ft - b
    arr = '↑' if d > 0 else ('↓' if d < 0 else '→')
    print(f"  {label:<22} {b:>7.1f}% {ft:>7.1f}% {arr}{abs(d):>5.1f}%")
 
print(f"  {'-'*55}")
print(f"  {'avg tokens':<22} {bm['avg_tokens']:>8.1f} {ft_metrics['avg_tokens']:>8.1f}")
print(f"{'='*65}")
 
# Urgency детально
print(f"\n📊 URGENCY ДЕТАЛЬНО:")
urgency_by_level = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in ft_results:
    exp = r['expected']['urgency']
    urgency_by_level[exp]['total'] += 1
    if r['scores']['urgency_correct']:
        urgency_by_level[exp]['correct'] += 1
 
for level in ['critical', 'high', 'medium', 'low']:
    d = urgency_by_level[level]
    if d['total'] > 0:
        pct = d['correct']/d['total']*100
        bar = '█' * int(pct/10)
        print(f"  {level:10} {d['correct']}/{d['total']} = {pct:.0f}% {bar}")
 
# Список всіх збережених файлів
print(f"\n📁 ЗБЕРЕЖЕНІ АРТЕФАКТИ:")
for fname in sorted(os.listdir(BASE_DIR)):
    fpath = os.path.join(BASE_DIR, fname)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)/1024
        print(f"   {fname:<35} {size:.1f} KB")
    elif os.path.isdir(fpath):
        dir_size = sum(os.path.getsize(os.path.join(dp,f))
                      for dp,_,files in os.walk(fpath) for f in files)
        print(f"   {fname}/ (dir)              {dir_size/1024/1024:.1f} MB")
 
print(f"\n🎯 ГОТОВО! Всі артефакти збережено.")
print(f"   Наступний крок — README.md")



  ФІНАЛЬНЕ ПОРІВНЯННЯ: Base 8B vs Fine-tuned 8B
  Метрика                    Base       FT        Δ
  -------------------------------------------------------
  json_valid               100.0%   100.0% →  0.0%
  exact_match               33.3%    56.7% ↑ 23.4%
  sender accuracy           90.0%   100.0% ↑ 10.0%
  product accuracy          63.3%    76.7% ↑ 13.4%
  category accuracy         76.7%    96.7% ↑ 20.0%
  urgency accuracy          63.3%    76.7% ↑ 13.4%
  summary present          100.0%   100.0% →  0.0%
  -------------------------------------------------------
  avg tokens                125.1     56.3

📊 URGENCY ДЕТАЛЬНО:
  critical   6/7 = 86% ████████
  high       8/9 = 89% ████████
  medium     1/4 = 25% ██
  low        8/10 = 80% ████████

📁 ЗБЕРЕЖЕНІ АРТЕФАКТИ:
   baseline_metrics.json               0.2 KB
   baseline_results.json               38.3 KB
   checkpoints/ (dir)              0.0 MB
   eval_hashes.json                    1.4 KB
   ft_metrics.json                

# Llama 3.3 70B Baseline через OpenRouter API
## Порівняння з нашим fine-tuned 8B
## OpenRouter: https://openrouter.ai

In [1]:
# ────────────────────────────────────────────────────────────
# 1: Встановлення + налаштування
# ────────────────────────────────────────────────────────────
!pip install -q openai
 
import os, json, re, time
from collections import defaultdict
from kaggle_secrets import UserSecretsClient
 
# Додай OPENROUTER_API_KEY в Kaggle Secrets
secrets            = UserSecretsClient()
OPENROUTER_API_KEY = secrets.get_secret("OPENROUTER_API_KEY")
 
BASE_DIR  = '/kaggle/working/lesson17_finetune'
EVAL_PATH = '/kaggle/input/datasets/alexandr2017/eval-set/eval_set.jsonl'
 
print("✅ OpenRouter API готовий")


✅ OpenRouter API готовий


In [2]:
# ────────────────────────────────────────────────────────────
# 2: Функції інференсу та оцінки
# ────────────────────────────────────────────────────────────
from openai import OpenAI
 
client = OpenAI(
    base_url = "https://openrouter.ai/api/v1",
    api_key  = OPENROUTER_API_KEY,
)
 
# Той самий промпт що і в baseline/fine-tuned
SYSTEM_PROMPT = """You are a support email classifier. Extract information from the email and return ONLY a valid JSON object with exactly these fields:
- sender: full name of the sender (null if anonymous)
- product: one of [Payments, Inventory, Analytics, Storefront, Shipping, Subscriptions]
- category: one of [billing, technical, account, feature_request, other]
- urgency: one of [low, medium, high, critical]
- summary: one sentence summary of the issue
 
Return ONLY the JSON object, no explanation, no markdown, no extra text."""
 
def run_groq_inference(email_text, model="meta-llama/llama-3.3-70b-instruct"):
    """Запускаємо 70B через OpenRouter API"""
    start = time.time()
    response = client.chat.completions.create(
        model    = model,
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Email:\n{email_text}"}
        ],
        temperature      = 0,        # greedy — аналог do_sample=False
        max_tokens       = 256,
 
    )
    latency     = time.time() - start
    raw         = response.choices[0].message.content
    input_tok   = response.usage.prompt_tokens
    output_tok  = response.usage.completion_tokens
    return raw, output_tok, latency, input_tok
 
def parse_json_response(raw):
    raw = raw.strip()
    try: return json.loads(raw)
    except: pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except: pass
    match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
    if match:
        try: return json.loads(match.group(0))
        except: pass
    return None
 
def score_prediction(pred, expected):
    if pred is None:
        return {k: 0 for k in ['json_valid','sender_correct','product_correct',
                                'category_correct','urgency_correct',
                                'summary_present','exact_match']}
    sender_correct = (
        (pred.get('sender') is None and expected.get('sender') is None) or
        (str(pred.get('sender','')).strip().lower() ==
         str(expected.get('sender','')).strip().lower())
    )
    product_correct  = pred.get('product')  == expected.get('product')
    category_correct = pred.get('category') == expected.get('category')
    urgency_correct  = pred.get('urgency')  == expected.get('urgency')
    summary_present  = bool(pred.get('summary','').strip())
    exact_match      = all([sender_correct, product_correct,
                            category_correct, urgency_correct, summary_present])
    return {
        'json_valid'       : 1,
        'sender_correct'   : int(sender_correct),
        'product_correct'  : int(product_correct),
        'category_correct' : int(category_correct),
        'urgency_correct'  : int(urgency_correct),
        'summary_present'  : int(summary_present),
        'exact_match'      : int(exact_match),
    }
 
print("✅ Функції готові")
 


✅ Функції готові


In [4]:
# ────────────────────────────────────────────────────────────
# 3: Evaluation Llama 3.3 70B на eval set
# ~3-5 хвилин (API calls з rate limiting)
# ────────────────────────────────────────────────────────────
import os
BASE_DIR = '/kaggle/working/lesson17_finetune'
os.makedirs(BASE_DIR, exist_ok=True)

with open(EVAL_PATH, 'r', encoding='utf-8') as f:
    eval_data = [json.loads(line) for line in f]
 
results_70b   = []
total_tokens  = 0
total_latency = 0
total_input_tokens = 0
 
print(f"🚀 Llama 3.3 70B evaluation на {len(eval_data)} прикладах...\n")
 
for i, item in enumerate(eval_data, 1):
    try:
        raw, out_tokens, latency, in_tokens = run_groq_inference(item['email'])
        pred   = parse_json_response(raw)
        scores = score_prediction(pred, item['expected'])
 
        total_tokens       += out_tokens
        total_latency      += latency
        total_input_tokens += in_tokens
 
        results_70b.append({
            'id'      : item['id'],
            'raw'     : raw,
            'parsed'  : pred,
            'expected': item['expected'],
            'scores'  : scores,
            'tokens'  : out_tokens,
            'latency' : round(latency, 2),
        })
 
        status     = '✅' if scores['json_valid']      else '❌'
        urgency_ok = '✅' if scores['urgency_correct'] else '❌'
        pred_urg   = pred.get('urgency','—') if pred else 'NONE'
        exp_urg    = item['expected']['urgency']
        print(f"  {i:2}/30 {item['id']} | JSON:{status} | urgency:{urgency_ok} "
              f"| pred:{pred_urg:8} | expected:{exp_urg} | {latency:.1f}s")
 
        # Rate limiting — невелика пауза між запитами
        time.sleep(1)
 
    except Exception as e:
        print(f"  {i:2}/30 {item['id']} | ERROR: {e}")
        results_70b.append({
            'id': item['id'], 'raw': '', 'parsed': None,
            'expected': item['expected'],
            'scores': score_prediction(None, item['expected']),
            'tokens': 0, 'latency': 0,
        })
        time.sleep(5)  # довша пауза після помилки
 
# Зберігаємо
with open(os.path.join(BASE_DIR, 'results_70b.json'), 'w') as f:
    json.dump(results_70b, f, ensure_ascii=False, indent=2)
 
print(f"\n💾 Результати збережено")
 


🚀 Llama 3.3 70B evaluation на 30 прикладах...

   1/30 eval_001 | JSON:✅ | urgency:✅ | pred:critical | expected:critical | 1.6s
   2/30 eval_002 | JSON:✅ | urgency:✅ | pred:high     | expected:high | 2.3s
   3/30 eval_003 | JSON:✅ | urgency:✅ | pred:low      | expected:low | 1.5s
   4/30 eval_004 | JSON:✅ | urgency:❌ | pred:high     | expected:medium | 2.5s
   5/30 eval_005 | JSON:✅ | urgency:✅ | pred:low      | expected:low | 1.8s
   6/30 eval_006 | JSON:✅ | urgency:✅ | pred:high     | expected:high | 2.1s
   7/30 eval_007 | JSON:✅ | urgency:✅ | pred:low      | expected:low | 1.4s
   8/30 eval_008 | JSON:✅ | urgency:✅ | pred:medium   | expected:medium | 2.1s
   9/30 eval_009 | JSON:✅ | urgency:❌ | pred:high     | expected:critical | 3.2s
  10/30 eval_010 | JSON:✅ | urgency:✅ | pred:low      | expected:low | 7.4s
  11/30 eval_011 | JSON:✅ | urgency:✅ | pred:critical | expected:critical | 1.4s
  12/30 eval_012 | JSON:✅ | urgency:✅ | pred:medium   | expected:medium | 0.9s
  13/30 eval_01

In [6]:
import os, json
BASE_DIR = '/kaggle/working/lesson17_finetune'
os.makedirs(BASE_DIR, exist_ok=True)

# Метрики з exp 1 (зафіксовані раніше)
bm = {
    "model"             : "Llama-3.1-8B-base",
    "json_valid"        : 100.0,
    "exact_match"       : 30.0,
    "sender_accuracy"   : 90.0,
    "product_accuracy"  : 63.3,
    "category_accuracy" : 80.0,
    "urgency_accuracy"  : 60.0,
    "summary_present"   : 100.0,
    "avg_tokens"        : 145.8,
}

fm = {
    "model"             : "Llama-3.1-8B-finetuned",
    "json_valid"        : 100.0,
    "exact_match"       : 56.7,
    "sender_accuracy"   : 100.0,
    "product_accuracy"  : 76.7,
    "category_accuracy" : 93.3,
    "urgency_accuracy"  : 76.7,
    "summary_present"   : 100.0,
    "avg_tokens"        : 56.1,
}

# Зберігаємо щоб не загубились знову
with open(os.path.join(BASE_DIR, 'baseline_metrics.json'), 'w') as f:
    json.dump(bm, f, indent=2)
with open(os.path.join(BASE_DIR, 'ft_metrics.json'), 'w') as f:
    json.dump(fm, f, indent=2)

print("✅ Метрики відновлено")

✅ Метрики відновлено


In [7]:
# ────────────────────────────────────────────────────────────
# 4: Фінальне порівняння всіх трьох моделей
# ────────────────────────────────────────────────────────────
n      = len(results_70b)
totals = defaultdict(int)
for r in results_70b:
    for k, v in r['scores'].items():
        totals[k] += v
 
avg_tokens_70b  = total_tokens / n
avg_latency_70b = total_latency / n
 
metrics_70b = {
    "model"             : "Llama-3.3-70B (Groq API)",
    "json_valid"        : round(totals['json_valid']/n*100, 1),
    "exact_match"       : round(totals['exact_match']/n*100, 1),
    "sender_accuracy"   : round(totals['sender_correct']/n*100, 1),
    "product_accuracy"  : round(totals['product_correct']/n*100, 1),
    "category_accuracy" : round(totals['category_correct']/n*100, 1),
    "urgency_accuracy"  : round(totals['urgency_correct']/n*100, 1),
    "summary_present"   : round(totals['summary_present']/n*100, 1),
    "avg_tokens"        : round(avg_tokens_70b, 1),
    "avg_latency_sec"   : round(avg_latency_70b, 2),
}
 
with open(os.path.join(BASE_DIR, 'metrics_70b.json'), 'w') as f:
    json.dump(metrics_70b, f, indent=2)
 
# Завантажуємо метрики базової і fine-tuned 8B
with open(os.path.join(BASE_DIR, 'baseline_metrics.json')) as f:
    bm = json.load(f)
with open(os.path.join(BASE_DIR, 'ft_metrics.json')) as f:
    fm = json.load(f)
 
metrics_order = [
    ('json_valid',        'json_valid'),
    ('exact_match',       'exact_match'),
    ('sender accuracy',   'sender_accuracy'),
    ('product accuracy',  'product_accuracy'),
    ('category accuracy', 'category_accuracy'),
    ('urgency accuracy',  'urgency_accuracy'),
    ('summary present',   'summary_present'),
]
 
print(f"\n{'='*75}")
print(f"  ФІНАЛЬНЕ ПОРІВНЯННЯ: Base 8B vs Fine-tuned 8B vs Llama 3.3 70B")
print(f"{'='*75}")
print(f"  {'Метрика':<22} {'Base 8B':>9} {'FT 8B':>9} {'70B':>9}")
print(f"  {'-'*60}")
 
for label, key in metrics_order:
    b  = bm.get(key, bm.get(key.replace('_accuracy','_correct'), 0))
    ft = fm.get(key, fm.get(key.replace('_accuracy','_correct'), 0))
    s  = metrics_70b[key]
    # Виділяємо найкращий результат
    best = max(b, ft, s)
    def fmt(v): return f"{'→' if v==best else ' '}{v:>7.1f}%"
    print(f"  {label:<22} {fmt(b)} {fmt(ft)} {fmt(s)}")
 
print(f"  {'-'*60}")
print(f"  {'avg tokens':<22} {bm['avg_tokens']:>9.1f} {fm['avg_tokens']:>9.1f} {avg_tokens_70b:>9.1f}")
print(f"  {'avg latency (sec)':<22} {'~1.1':>9} {'~1.1':>9} {avg_latency_70b:>9.2f}")
print(f"{'='*75}")
 
# Urgency детально для 70B
print(f"\n📊 URGENCY ДЕТАЛЬНО — Llama 3.3 70B:")
urgency_by_level = defaultdict(lambda: {'correct': 0, 'total': 0})
for r in results_70b:
    exp = r['expected']['urgency']
    urgency_by_level[exp]['total'] += 1
    if r['scores']['urgency_correct']:
        urgency_by_level[exp]['correct'] += 1
 
print(f"  {'Level':<12} {'Base 8B':>8} {'FT 8B':>8} {'70B':>8}")
print(f"  {'-'*40}")
base_urg = {'critical':'?/7', 'high':'?/9', 'medium':'?/4', 'low':'?/10'}
ft_urg   = {'critical':'6/7', 'high':'8/9', 'medium':'1/4', 'low':'8/10'}
for level in ['critical', 'high', 'medium', 'low']:
    d   = urgency_by_level[level]
    pct = f"{d['correct']}/{d['total']}"
    print(f"  {level:<12} {base_urg[level]:>8} {ft_urg[level]:>8} {pct:>8}")
 
# Cost analysis
print(f"\n💰 COST ANALYSIS (50,000 emails/день):")
print(f"  {'Модель':<30} {'Cost/місяць':>15} {'Latency':>10}")
print(f"  {'-'*60}")
print(f"  {'Llama 3.3 70B (Together AI)':<30} {'~$180':>15} {'2-3 сек':>10}")
print(f"  {'Llama 3.1 8B base (API)':<30} {'~$45':>15} {'1-2 сек':>10}")
print(f"  {'Fine-tuned 8B (Kaggle/Colab)':<30} {'~$0-30':>15} {'~1.1 сек':>10}")
print(f"  {'Fine-tuned 8B (AWS g4dn.xlarge)':<30} {'~$90':>15} {'~1.1 сек':>10}")
print(f"\n  Training cost: $0 (Kaggle free T4)")
print(f"  Inference speedup vs 70B: ~2-3x faster")
 
print(f"\n🎯 Гіпотеза курсу:")
ft_urg_acc  = fm['urgency_accuracy']
b70_urg_acc = metrics_70b['urgency_accuracy']
if ft_urg_acc >= b70_urg_acc:
    print(f"  ✅ ПІДТВЕРДЖЕНА: Fine-tuned 8B ({ft_urg_acc}%) ≥ 70B ({b70_urg_acc}%) по urgency accuracy")
else:
    print(f"  ⚠️  ЧАСТКОВО: Fine-tuned 8B ({ft_urg_acc}%) < 70B ({b70_urg_acc}%) по urgency accuracy")
    print(f"     Але: 8B значно дешевший і швидший для inference")



  ФІНАЛЬНЕ ПОРІВНЯННЯ: Base 8B vs Fine-tuned 8B vs Llama 3.3 70B
  Метрика                  Base 8B     FT 8B       70B
  ------------------------------------------------------------
  json_valid             →  100.0% →  100.0% →  100.0%
  exact_match                30.0% →   56.7%     50.0%
  sender accuracy            90.0% →  100.0% →  100.0%
  product accuracy           63.3% →   76.7%     73.3%
  category accuracy          80.0% →   93.3%     73.3%
  urgency accuracy           60.0%     76.7% →   80.0%
  summary present        →  100.0% →  100.0% →  100.0%
  ------------------------------------------------------------
  avg tokens                 145.8      56.1      53.3
  avg latency (sec)           ~1.1      ~1.1      2.39

📊 URGENCY ДЕТАЛЬНО — Llama 3.3 70B:
  Level         Base 8B    FT 8B      70B
  ----------------------------------------
  critical          ?/7      6/7      3/7
  high              ?/9      8/9      9/9
  medium            ?/4      1/4      3/4
  low     